# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIRˆ<sup>2</sup> dataset using the `mlcroissant` library, referencing Croissant entities by their `@id`.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure required libraries are installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {', '.join(str(k) for k in getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets and their fields referenced by `@id`.

Let's retrieve all record sets described in the schema, their `@id`, and summarize their fields and columns.

In [ ]:
# List all record sets with their @id and field columns
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
for rset in record_sets:
    print(f"@id: {rset['@id']} | name: {rset.get('name', '')}")
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Field @id(s):", [f['@id'] for f in fields])
    print()
# Save the first record set @id as example
if record_sets:
    first_record_set_id = record_sets[0]['@id']
else:
    first_record_set_id = None

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing record set and field `@id`s.

In [ ]:
# Build a list of all record set @id values
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set '@id': {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded shape: {df.shape}")
        print(f"Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}. Error: {e}\n")

# Display the first few rows of the first available record set DataFrame
if first_record_set_id and first_record_set_id in dataframes:
    df = dataframes[first_record_set_id]
    print(f"First rows of record set '@id': {first_record_set_id}")
    display(df.head())
else:
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, and grouping, referencing fields by their `@id` (column names).

In [ ]:
import numpy as np
# Identify a numeric field/column in the DataFrame
if df is not None:
    # Try to find a likely numeric field (fallback to first float-like column)
    numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]
    if not numeric_candidates:
        # Try to infer numeric columns by converting
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    print(f"Numeric candidate fields: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        # Filtering (arbitrary threshold for demonstration)
        threshold = df[numeric_field].mean()  # Use mean as threshold for demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Grouping (try to group by any categorical column with low cardinality)
        candidates = [c for c in df.columns if df[c].dtype == object and df[c].nunique() < 10]
        if candidates:
            group_field = candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df)
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions and relationships from the selected record set, always referencing fields by column name (`@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If possible, boxplot by a group field
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded and explored the Croissant-based FAIR⁲ dataset using `mlcroissant`.
- Identified available record sets and their fields via their `@id`.
- Loaded record set data to DataFrame, performed basic data analysis and filtering referencing field `@id`s.
- Visualized the distribution and grouping in selected numeric columns.

For deeper analysis, refer to the [FAIR2 package documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and explore additional metadata, provenance, or quality information encoded in the schema via their unique `@id` references.